# Les composants de pommes_craft et leurs défauts

*Pour chaque composant de pommes_craft 0.6.0, ce qu'il représente, ses principaux arguments avec leurs unités et leurs défauts lus dans le code, le paramètre pommes qu'il alimente et son piège ; cinq hypothèses traduites en composant ; une batterie de 1 MW / 2 h ajoutée au modèle du tutoriel.*

Ce notebook est tiré du texte [en ligne](https://www.robingirard.eu/assets/parcours-es/latest/textes/fiche-composants-craft.html) du parcours ES ; exécutez les cellules dans l'ordre (Maj + Entrée). Les commentaires des cellules donnent les valeurs attendues.

**La cellule d'installation.** En local, placez ce notebook dans le dépôt `pommes-case-studies` (à sa racine ou dans un de ses dossiers) et ouvrez-le avec le noyau du venv du dépôt (`.venv`), comme l'explique la page [Installer et vérifier](https://www.robingirard.eu/assets/parcours-es/latest/textes/installer.html) : la cellule n'installe rien, elle trouve le dépôt et affiche les versions des paquets. Une ligne « Attention » signale un autre noyau : changez-le (Kernel › Change Kernel).

Sur Google Colab (Fichier › Importer un notebook), la cellule clone le dépôt et installe les paquets : 2 à 3 minutes, puis elle s'arrête volontairement sur `RedemarrezLaSession`. Faites Exécution › Redémarrer la session, relancez cette cellule, et seulement elle, puis continuez dans l'ordre.

In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

VERSIONS = {"pommes": "0.11.0", "pommes-craft": "0.6.0", "linopy": "0.10.1", "highspy": "1.15.1"}


def versions_du_depot() -> bool:
    """Vrai si les paquets installés sont ceux de requirements.txt (les valeurs attendues en dépendent)."""
    try:
        return all(version(nom) == v for nom, v in VERSIONS.items())
    except PackageNotFoundError:
        return False


class RedemarrezLaSession(Exception):
    """Colab : les paquets sont installés, mais la session garde en mémoire les anciens numpy et pandas."""


if "google.colab" in sys.modules:                     # sur Colab : cloner, installer, redémarrer une fois
    CLONE = Path("/content/pommes-case-studies")
    if not CLONE.exists():
        !git clone https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies {CLONE}
    %cd {CLONE}
    if not versions_du_depot():
        !python --version
        !pip install -q -r requirements.txt
        raise RedemarrezLaSession("Installation finie. Exécution › Redémarrer la session, "
                                  "puis relancez cette cellule (et seulement elle).")

RACINE = next((d for d in (Path.cwd(), *Path.cwd().parents, Path.cwd() / "pommes-case-studies")
               if (d / "pommes_cases").is_dir()), None)  # la racine du dépôt des cas
if RACINE is None:
    raise FileNotFoundError(
        "dépôt pommes-case-studies introuvable : ce notebook a besoin de son code et de ses données. "
        "Clonez le dépôt, placez ce notebook dedans (à sa racine ou dans un de ses dossiers) et ouvrez-le avec "
        "le noyau du venv du dépôt (https://www.robingirard.eu/assets/parcours-es/latest/textes/installer.html). "
        "Sans rien installer : ouvrez-le sur Google Colab.")
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))
print("dépôt :", RACINE.name, "· Python", sys.version.split()[0])
print(" · ".join(f"{nom} {version(nom)}" for nom in VERSIONS))
if not versions_du_depot():
    print("Attention : ce ne sont pas les versions du dépôt", VERSIONS, ": les valeurs attendues peuvent bouger.")
    print("Le noyau n'est sans doute pas celui du venv du dépôt : Kernel › Change Kernel.")

**Repris du notebook du socle** : le modèle de base, le PV et la turbine (étapes 2 à 4), sans les résolutions.

In [ ]:
from pommes_cases.prices import load_prices, price_sets

p = load_prices("tyndp2026", "NT+", "FR", 2030)
prix = p["import_price"]


import logging

import polars as pl
from pommes_craft import (Area, ConversionTechnology, Demand, EconomicHypothesis, EnergyModel, LoadShedding,
                          NetImport, Spillage, TimeStepManager)

logging.getLogger().setLevel(logging.ERROR)  # tait l'écho des paramètres laissés à leur valeur par défaut


def modele_de_base(nom, delestage="interdit"):
    """delestage : "interdit" (capacité 0), "aucun" (pas de composant) ou un coût en €/MWh."""
    m = EnergyModel(name=nom, hours=list(range(8760)), year_ops=[2030], year_invs=[2030], year_decs=[2055],
                    modes=[], resources=["electricity"], solver_name="highs",
                    solver_options={"output_flag": False})
    with m.context():
        EconomicHypothesis("eco", discount_rate=0.0, year_ref=2030, planning_step=1)
        TimeStepManager("pas", time_step_duration=1.0, operation_year_duration=8760.0)
        zone = Area("FR")
        zone.add_component(Demand(name="demande", resource="electricity",
                                  demand=pl.DataFrame({"hour": range(8760), "year_op": 2030, "demand": 1.0})))
        if delestage == "interdit":
            zone.add_component(LoadShedding(name="delestage", resource="electricity", max_capacity=0.0))
        elif delestage != "aucun":
            zone.add_component(LoadShedding(name="delestage", resource="electricity", cost=float(delestage)))
        zone.add_component(Spillage(name="ecretement", resource="electricity", max_capacity=0.0))
        zone.add_component(NetImport(name="reseau", resource="electricity",
                                     import_price=p.select("hour", "year_op", "import_price"),
                                     max_yearly_energy_export=0.0))
    return m, zone


from pommes_cases.meteo import availability, load_meteo

pv_dispo = availability(load_meteo("FR", 2030), "pv")


def pv():
    return ConversionTechnology(name="pv", factor={"electricity": 1.0}, availability=pv_dispo,
                                invest_cost=600_000.0, life_span=25, finance_rate=0.04)


def turbine():
    return ConversionTechnology(name="turbine_gaz", factor={"electricity": 1.0}, variable_cost=110.0,
                                invest_cost=0.0, life_span=25,
                                power_capacity_investment_min=1.0, power_capacity_investment_max=1.0)

Un modèle pommes_craft est une liste de composants déclarés dans `with model.context():`. Chaque composant devient un ou plusieurs paramètres pommes, préfixés par son module (`conversion_`, `storage_`, `net_import_`...). Deux jeux de défauts se superposent, qui ne coïncident pas toujours : ceux de la signature pommes_craft, dès que le composant existe, et ceux de pommes, quand il n'existe pas. Défauts lus dans le code installé de pommes_craft 0.6.0 (`inspect.signature`) et dans `dataset_description.yaml` de pommes 0.11.0 ; tableaux non exhaustifs (annuité, déclassement anticipé et quelques bornes omis). Unités : nomenclature pommes, MW, MWh, €, kg de CO₂. Pièges : [Les pièges connus de pommes et pommes_craft : symptôme, cause, remède](https://www.robingirard.eu/assets/parcours-es/latest/textes/pieges-pommes.html).

## Le cadre : EnergyModel et trois composants sans zone

`EnergyModel` porte les indices ; ses arguments s'écrivent nommés, jamais en positionnel. Une capacité investie en `year_inv` n'existe que si `year_inv + life_span` est une étiquette de `year_decs` (piège 3).

| Composant | Ce qu'il représente | Arguments et défauts | Paramètre pommes |
|---|---|---|---|
| `EconomicHypothesis` | Le temps et l'argent | `discount_rate=0.0` (fraction), `year_ref=2020`, `planning_step=nan`, `operation_year_weight=1.0` | `discount_rate`, `year_ref`, `planning_step`, `operation_year_weight` |
| `TimeStepManager` | Le pas et l'année | `time_step_duration=nan` (h par pas), `operation_year_duration=nan` (h par an ; docstring « days » trompeuse : écrire 8760.0) | `time_step_duration`, `operation_year_duration` |
| `Carbon` | Taxe ou plafond CO₂ | `tax=nan` (€/kg), `goal=nan` (kg/an) ; déclaré hors zone, il s'applique à toutes les zones (`carbon_tax` est pourtant indexé `area, year_op` dans pommes) | `carbon_tax`, `carbon_goal` |

Piège 6 : une taxe de 45 €/t s'écrit `tax=0.045`. Vérifié sur deux zones de 24 h : 24 MWh de gaz à 202 kg/MWh et 0,045 €/kg ajoutent 24 × 202 × 0,045 = 218,16 € (`operation_carbon_costs`).

## Les composants d'une zone

`Area(name)` est la zone ; `area.add_component(...)` y range tout ce qui suit.

| Composant | Ce qu'il représente | Principaux arguments et défauts | Paramètre pommes | Piège |
|---|---|---|---|---|
| `Demand` | Une consommation rigide | `resource`, `demand=0.0` (MW ; float, dict par heure ou DataFrame `hour, year_op, demand`) | `demand` | Un dict est lu par heure : `demand={2030: 1.0}` lève `ValueError: Invalid hour values`. Le DataFrame doit couvrir toutes les heures |
| `LoadShedding` | Le délestage, par ressource | `cost=1000.0` (€/MWh), `max_capacity=nan` (MW) | `load_shedding_cost`, `load_shedding_max_capacity` | Sans composant, pommes met coût 0 et capacité illimitée (piège 1) |
| `Spillage` | L'écrêtement, par ressource | `cost=1000.0`, `max_capacity=0.0` : interdit par défaut | `spillage_cost`, `spillage_max_capacity` | Sans composant, pommes le rend gratuit et illimité |
| `NetImport` | Le marché ou un fournisseur extérieur | `resource`, `import_price=0.0`, `export_price=0.0` (€/MWh, float ou DataFrame horaire), `max_yearly_energy_import=nan`, `max_yearly_energy_export=nan` (MWh/an), `emission_factor=0.0` (kg/MWh) | `net_import_import_price`, `net_import_export_price`, `net_import_max_yearly_energy_*` | Un seul par zone et ressource ; prix 0 ; ouvre toutes les ressources ; aucune borne horaire (piège 2) |
| `ConversionTechnology` | Une centrale ou un convertisseur | `factor` (requis, sans dimension : MWh de ressource par MWh d'activité de la technologie ; la ressource consommée est négative), `availability=nan` (fraction horaire), `invest_cost=0.0` (€/MW), `fixed_cost=0.0` (€/MW/an), `variable_cost=0.0` (€/MWh), `finance_rate=0.0`, `life_span=5` (ans), `emission_factor=0.0` (kg/MWh), `power_capacity_investment_min/max=nan`, `must_run`, `ramp_up`, `ramp_down`, `max_yearly_production=nan` | `conversion_*` | `life_span=5` par défaut : avec `year_decs=[2055]` la capacité disparaît sans message (piège 3) ; si `power_capacity_investment_min` l'impose, pommes 0.11.0 lève une `ValueError` |
| `StorageTechnology` | Batterie, STEP, réservoir | `factor_in`, `factor_keep`, `factor_out` (requis, par ressource), `dissipation=0.0`, `invest_cost_power=0.0` (€/MW), `invest_cost_energy=0.0` (€/MWh), `fixed_cost_power/energy=0.0`, `finance_rate=0.0`, `life_span=5`, `power_capacity_investment_min/max=nan`, `energy_capacity_investment_min/max=nan`, `energy_to_power_ratio=nan` (h) | `storage_*` | Le même `life_span=5` ; `factor_out` à 0 dans un exemple de la doc |
| `FlexibleDemand` | Une demande déplaçable | `resource`, `demand=nan` (MW), `conservation_hrs=0` (h), `min_demand`, `max_demand=nan` (MW), `ramp_up`, `ramp_down`, `variable_cost=nan` (€/MWh déplacé) | `flexibility_*` | `conservation_hrs=0` : conservation sur tout l'horizon, la demande peut se déplacer d'un bout de l'année à l'autre. Pommes impose toujours la somme annuelle ; les fenêtres ne s'ajoutent que si `conservation_hrs > 0` |
| `CombinedTechnology` | Une unité à plusieurs modes (cogénération) | comme `ConversionTechnology`, sans `availability` ; `factor` et `variable_cost` par mode ; `modes` déclarés dans `EnergyModel` | `combined_*` | |
| `Turpe` | Le tarif de réseau français | `calendar=nan` (type d'heure par heure), `fixed_cost=0.0`, `variable_cost=0.0` par `hour_type` | `turpe_*` | `hour_types` à déclarer dans `EnergyModel` ; hors zone, il s'applique à toutes |

Vérifié pour `FlexibleDemand` sur 48 h, prix 100 €/MWh puis 10 €/MWh, demande 1 MW, `max_demand=2.0` : `conservation_hrs=0` donne 480 €, les 48 MWh consommés dans les 24 heures à 10 € ; `conservation_hrs=24` ou `1` donne 2 640 €, comme une `Demand` rigide.

Trois composants de plus existent : `ProductionGroup` (plafond annuel partagé), `RetrofitConversion` et `RetrofitCombined` (conversion d'une technologie en une autre).

## Entre deux zones : Link et TransportTechnology

`Link(name, area_from, area_to)` relie deux `Area` ; `link.add_transport_technology(TransportTechnology(...))` y pose une ligne. Arguments : `resource` (requis), `invest_cost=0.0` (€/MW), `life_span=5`, `power_capacity_investment_min/max=nan` (MW), `hurdle_costs=nan` (€/MWh transporté), `finance_rate=nan`, `fixed_cost=nan`. Les `nan` restent `nan` dans les paramètres pommes (`transport_hurdle_costs`, lu après `store_pommes_parameters=True`) et comptent pour 0 dans l'objectif. Vérifié sur deux zones de 24 h : une ligne de 1 MW transporte 24 MWh pour 1 200 € de gaz à 50 €/MWh ; `hurdle_costs=2.0` ajoute 48 €, rangés dans `operation_transport_costs` depuis pommes 0.11. L'activité [Deux zones et un lien : flux, duale de la capacité d'échange et rente de congestion](https://www.robingirard.eu/assets/parcours-es/latest/textes/act-transport-deux-zones.html) va plus loin.

## Cinq phrases d'hypothèse

| Phrase | Composant et argument | Paramètre pommes |
|---|---|---|
| « 20 % de la demande est déplaçable dans la journée » | `FlexibleDemand(demand=0.2 × profil, conservation_hrs=24, max_demand=..., variable_cost=...)` ; le reste en `Demand` | `flexibility_demand`, `flexibility_conservation_hrs` |
| « Le gaz est acheté à 35 €/MWh » | `NetImport(resource="methane", import_price=35.0, max_yearly_energy_export=0.0)` et une `ConversionTechnology` dont `factor["methane"]` est négatif | `net_import_import_price` |
| « La capacité d'échange avec le voisin est de 3 GW » | `TransportTechnology(power_capacity_investment_min=3000.0, power_capacity_investment_max=3000.0)` sur un `Link` ; face à un prix exogène, `NetImport` ne borne que l'énergie annuelle, pas la puissance | `transport_power_capacity_investment_*` |
| « Le délestage est interdit » | `LoadShedding(resource, max_capacity=0.0)` | `load_shedding_max_capacity` |
| « Une taxe CO₂ de 80 €/t » | `Carbon(tax=0.080)` et `emission_factor` en kg/MWh sur chaque technologie et chaque `NetImport` | `carbon_tax`, `conversion_emission_factor` |

## Activité : une batterie de 1 MW / 2 h dans le modèle du tutoriel

Objectif : ajouter un stockage au modèle du bloc 4 de [Le premier modèle sur une page : les huit étapes du notebook et leurs valeurs attendues](https://www.robingirard.eu/assets/parcours-es/latest/textes/premier-modele.html) et lire son effet. Prérequis : ce tutoriel exécuté (497 405,29 €). Chaque résolution prend 2 à 3 secondes.

1. Reprendre `modele_de_base`, `pv()` et `turbine()` du tutoriel. Ajouter la batterie : rendement 0,9 à la décharge, taille imposée, investissement nul. `life_span=25` accorde la batterie à `year_decs=[2055]`.

In [ ]:
from pommes_craft import StorageTechnology
def batterie(**kw):
    base = dict(name="batterie", factor_in={"electricity": -1.0}, factor_out={"electricity": 0.9},
                factor_keep={"electricity": 0.0}, dissipation=0.0, life_span=25,
                power_capacity_investment_min=1.0, power_capacity_investment_max=1.0,
                energy_to_power_ratio=2.0)
    base.update(kw)
    return StorageTechnology(**base)

m, zone = modele_de_base("tuto_batterie")
with m.context():
    zone.add_component(pv()); zone.add_component(turbine()); zone.add_component(batterie())
m.run(store_linopy_model=True)
print(f"coût total {m.linopy_model.objective.value:,.2f} €")   # 472,144.28 €
print(m.get_results("planning", "power_capacity").select("name", "value"))   # pv 2.150538, turbine_gaz 1.0, batterie 1.0

2. Lire les cycles dans `operation_storage_net_generation` : positif quand la batterie injecte, négatif quand elle charge.

In [ ]:
sol = m.linopy_model.solution
ng = sol["operation_storage_net_generation"].sel(area="FR", resource="electricity").squeeze()
e_cap = float(sol["planning_storage_energy_capacity"].sum())                 # 2.0 MWh
injecte = float(ng.where(ng > 0, 0).sum()); charge = float(-ng.where(ng < 0, 0).sum())
print(f"injecté {injecte:.0f} MWh, chargé {charge:.0f} MWh, cycles {injecte / e_cap:.0f}")
# injecté 672 MWh, chargé 747 MWh, cycles 336
print(int((ng > 1e-6).sum()), int((ng < -1e-6).sum()))   # environ 839 heures d'injection, 1309 heures de charge

Le coût est unique, pas le détail horaire : 8 heures ont un prix nul et 196 sont sous 1 €/MWh, donc plusieurs plannings de la batterie coûtent exactement pareil. D'une exécution ou d'une version à l'autre, les heures d'injection et de charge peuvent varier de quelques unités, une douzaine au plus, et l'énergie de moins d'un MWh, au même coût de 472 144,28 €. Les cycles arrondis (336) et la perte (75 MWh) tiennent.

3. Le piège : remettre `life_span` au défaut, 5 ans, et retirer le plancher de puissance. La batterie peut encore aller jusqu'à 1 MW.

In [ ]:
m5, zone = modele_de_base("tuto_batterie_lifespan5")
with m5.context():
    zone.add_component(pv()); zone.add_component(turbine())
    zone.add_component(batterie(life_span=5, power_capacity_investment_min=float("nan")))
m5.run(store_linopy_model=True)
print(f"{m5.linopy_model.objective.value:,.2f} €", float(m5.linopy_model.solution["planning_storage_power_capacity"].sum()))
# 497,405.29 € 0.0 : la batterie a disparu, le modèle est celui du tutoriel

Avec le plancher, `batterie(life_span=5)` ne tourne plus : pommes 0.11.0 lève `ValueError: storage_power_capacity_investment_min = 1 at area=FR, storage_tech=batterie, year_inv=2030 is imposed on a vintage without planning variable` (contrôle ajouté en 0.10). Une capacité imposée est protégée ; une capacité laissée au choix de pommes disparaît toujours sans message.

À noter : le coût passe de 497 405,29 à 472 144,28 €, soit 25 261 € de moins pour 1 MW ; le PV monte de 1,647 à 2,151 MW, la turbine tombe de 181,6 à 144,8 MWh, l'import de 6 619 à 6 118 MWh. La batterie vaut en partie par le PV qu'elle rend possible. Si pommes choisit la taille (`invest_cost_power=100_000`, `invest_cost_energy=150_000`, `finance_rate=0.04`, bornes retirées), il prend 0,186 MW et le coût vaut 497 347,97 €.

Questions. Quelle énergie se perd entre charge et injection, et quel argument la porte ? (747 − 672 = 75 MWh, `factor_out=0.9`.) Combien de cycles par jour en moyenne ? (336 / 365 = 0,92.) Que vaut `planning_storage_energy_capacity` si l'on remplace `energy_to_power_ratio=2.0` par `energy_capacity_investment_min=2.0, energy_capacity_investment_max=2.0` ? (2,0 MWh, même coût.)

## Vous l'avez acquise quand…

- Pour les cinq phrases du tableau, vous nommez le composant, l'argument et son unité sans ouvrir la fiche.
- Vous retrouvez 472 144,28 € et 336 cycles, et vous expliquez les 25 261 € de gain par les trois mouvements du bilan.
- Vous dites, avant de lancer, ce que fait un `StorageTechnology` laissé à `life_span=5` avec `year_decs=[2055]`, avec et sans plancher de puissance, et ce que fait une `FlexibleDemand` laissée à `conservation_hrs=0`.
- Vous écrivez un gaz à prix fixe avec deux composants et un `factor` négatif.

## Pour aller plus loin

- Les valeurs par défaut de chaque paramètre pommes, avec ses dimensions [Dataset description : la table de tous les paramètres (défaut, dimensions, type)](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes/methodology/dataset_description.html).
- Les trois exemples de la doc pommes_craft : PV + gaz, deux zones reliées, batterie (site en 0.3.7 ; `factor_out` à 0 dans l'exemple batterie) [Examples de pommes_craft : PV + gaz, deux zones reliées, batterie](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes_craft/examples.html).
- Le cas A : la même batterie face au marché, revenu par kW et n-ième GW [Cas A — Batteries : que rapporte l'arbitrage au n-ième GW ?](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies/-/tree/main/A_batteries).
- Une demande déplaçable en trente lignes [Une FlexibleDemand de 30 lignes sur le modèle du tutoriel](https://www.robingirard.eu/assets/parcours-es/latest/textes/act-demande-flexible-craft.html).

## Sources

1. [pommes_craft 0.6.0 sur PyPI (code lu : pommes_craft/components/*.py, core/model.py)](https://pypi.org/project/pommes-craft/)
2. [Dépôt pommes_craft (RELEASE_NOTES, examples)](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_craft)
3. [Doc pommes, Dataset description (défauts de pommes 0.11.0, lus dans model/data_validation/dataset_description.yaml)](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes/methodology/dataset_description.html)
4. [Doc pommes, Model description (stockage, carbone, transport, demande flexible)](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes/methodology/model_description.html)
5. [Dépôt pommes-case-studies (A_batteries/model.py, pommes_cases/systeme_fr.py, pommes_cases/blocks.py)](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies)